In [ ]:
import glob, re, os
import xarray as xr, xesmf as xe, numpy as np
from tcpyPI.pi import pi

# from xclim.core.units import convert_units_to
from datetime import datetime

from  IPython.display import clear_output

import warnings
warnings.filterwarnings("ignore", category = FutureWarning)
warnings.filterwarnings("ignore", message = ".+multiple fill values.+")

# xn,xx,yn,yx = [-100,-20,0,30] # original Caribbean region
xn,xx,yn,yx = [-85,-35,5,25] # Melissa track only
rnm = "melissa"

In [ ]:
def wrap_lon(ds):
    
    # method to wrap longitude from (0,360) to (-180,180)
    if "longitude" in ds.coords: ds = ds.rename(longitude = "lon", latitude = "lat")
    
    if ds.lon.max() > 180:
        ds["lon"] = (ds.lon.dims, (((ds.lon.values + 180) % 360) - 180), ds.lon.attrs)
        
    ds = ds.reindex({ "lon" : np.sort(ds.lon) })
    ds = ds.reindex({ "lat" : np.sort(ds.lat) })
    
    return ds

# Identify candidate models

In [ ]:
# list models for each variables (first realisation only)
fl_tos = sorted(list(set(["/".join(fp.split("/")[6:10]) for fp in glob.glob("/badc/cmip6/data/CMIP6/ScenarioMIP/*/*/ssp585/r1i1p1f1/Omon/tos/*")])))
fl_ta = sorted(list(set([re.sub("ta","*","/".join(fp.split("/")[6:])) for fp in glob.glob("/badc/cmip6/data/CMIP6/ScenarioMIP/*/*/ssp585/r1i1p1f1/Amon/ta/*")])))
fl_psl = sorted(list(set([re.sub("psl","*","/".join(fp.split("/")[6:])) for fp in glob.glob("/badc/cmip6/data/CMIP6/ScenarioMIP/*/*/ssp585/r1i1p1f1/Amon/psl/*")])))
fl_hurs = sorted(list(set([re.sub("hurs","*","/".join(fp.split("/")[6:])) for fp in glob.glob("/badc/cmip6/data/CMIP6/ScenarioMIP/*/*/ssp585/r1i1p1f1/Amon/hurs/*")])))

# get list of models that appear in all lists (match grid across Amon variables, then match to Omon)
mlist_Amon = [fp for fp in fl_hurs if fp in fl_psl and fp in fl_ta]
mlist = [m for m in mlist_Amon if "/".join(m.split("/")[:4]) in fl_tos]

In [ ]:
# for models where r1i1p1f1 doesn't exist, get first ensemble member
tos_notr1 = ["/".join(fp.split("/")[6:10]) for fp in glob.glob("/badc/cmip6/data/CMIP6/ScenarioMIP/*/*/ssp585/*/Omon/tos/*") if not "r1i1p1f1" in fp]
ta_notr1 = [re.sub("ta","*","/".join(fp.split("/")[6:])) for fp in glob.glob("/badc/cmip6/data/CMIP6/ScenarioMIP/*/*/ssp585/*/Amon/ta/*") if not "r1i1p1f1" in fp]
psl_notr1 = [re.sub("psl","*","/".join(fp.split("/")[6:])) for fp in glob.glob("/badc/cmip6/data/CMIP6/ScenarioMIP/*/*/ssp585/*/Amon/psl/*") if not "r1i1p1f1" in fp]
hurs_notr1 = [re.sub("hurs","*","/".join(fp.split("/")[6:])) for fp in glob.glob("/badc/cmip6/data/CMIP6/ScenarioMIP/*/*/ssp585/*/Amon/hurs/*") if not "r1i1p1f1" in fp]

# get list of models that appear in all lists
mlist_notr1_Amon = [fp for fp in hurs_notr1 if fp in psl_notr1 and fp in ta_notr1]
mlist_notr1 = [m for m in mlist_notr1_Amon if "/".join(m.split("/")[:4]) in tos_notr1]

In [ ]:
# find models without r1
gcmlist_r1 = sorted(list(set(["/".join(m.split("/")[:3]) for m in mlist])))
gcmlist_notr1 = sorted(list(set(["/".join(m.split("/")[:3]) for m in mlist_notr1])))
gcmlist_notr1 = [m for m in gcmlist_notr1 if not m in gcmlist_r1]

# get first realisation
mlist_notr1_1 = [sorted([fp for fp in mlist_notr1 if m in fp])[0] for m in gcmlist_notr1]

# combine both lists to get all available realisations
mlist = mlist + mlist_notr1_1

excl = [
        'AWI/AWI-CM-1-1-MR/ssp585/r1i1p1f1',
        'CNRM-CERFACS/CNRM-CM6-1-HR/ssp585/r1i1p1f2',
        'FIO-QLNM/FIO-ESM-2-0/ssp585/r1i1p1f1',
        'HadGEM3-GC31-MM/ssp585/r1i1p1f3',
        'CESM2-WACCM/ssp585/r2i1p1f1'
       ]

mlist = sorted([m for m in mlist if not "/".join(m.split("/")[:4]) in excl])

# Compile MSLP

In [ ]:
for m in mlist:

    print(m, end = " ")
    mdl = m.split("/")[1]+"_"+m.split("/")[3]

    psl_fp = "/badc/cmip6/data/CMIP6/ScenarioMIP/"+re.sub("\*","psl",m)+"/latest/*"

    fl_hist = sorted(glob.glob(re.sub("ScenarioMIP", "CMIP", re.sub("ssp585", "historical", psl_fp))))
    fl_ssp = sorted([fnm for fnm in glob.glob(psl_fp) if int(fnm[-16:-12]) <= 2100])

    if len(fl_hist) == 0:
        print("No historical data")
        continue
    if len(fl_ssp) == 0: 
        print("No ssp585 data")
        continue

    new_fnm = "psl/psl_"+rnm+"_"+mdl+fl_hist[0][-17:-9]+fl_ssp[-1].split("/")[-1][-9:]
    if os.path.exists(new_fnm): 
        print("Already processed")
        continue

    ds_hist = [xr.open_dataset(fnm).psl.convert_calendar("standard", align_on = "date") for fnm in fl_hist]
    ds_ssp = [xr.open_dataset(fnm).psl.convert_calendar("standard", align_on = "date") for fnm in fl_ssp]

    ds = wrap_lon(xr.concat(ds_hist + ds_ssp, "time")).sel(lon = slice(xn,xx), lat = slice(yn,yx))

    if ds.units == "Pa":
        ds = (ds / 100).assign_attrs(units = "hPa")

    ds.to_netcdf(new_fnm)
    print("")
# clear_output(wait = False)
print("Done.")

# Regrid & compile SSTs

In [ ]:
for m in mlist:

    print(m, end = ": ")
    mdl = m.split("/")[1]+"_"+m.split("/")[3]
    
    tos_fp = sorted(glob.glob("/badc/cmip6/data/CMIP6/ScenarioMIP/"+"/".join(m.split("/")[:4])+"/Omon/tos/*"))[0]+"/latest/*"

    fl_hist = glob.glob(re.sub("ScenarioMIP", "CMIP", re.sub("ssp585", "historical", tos_fp)))
    fl_ssp = [fnm for fnm in glob.glob(tos_fp) if int(fnm[-16:-12]) <= 2100]

    if len(fl_hist) == 0:
        print("No historical data")
        continue
    if len(fl_ssp) == 0: 
        print("No ssp585 data")
        continue

    new_fnm = "tos/tos_"+rnm+"_"+mdl+fl_hist[0][-17:-9]+fl_ssp[-1].split("/")[-1][-9:]
    if os.path.exists(new_fnm): 
        print("Already processed")
        continue

    print("loading... ", end = "")
    ds_hist = [xr.open_dataset(fnm).tos.convert_calendar("standard", align_on = "date") for fnm in fl_hist]
    ds_ssp = [xr.open_dataset(fnm).tos.convert_calendar("standard", align_on = "date") for fnm in fl_ssp]

    print("concatenating... ", end = "")
    ds = xr.concat(ds_hist + ds_ssp, "time")

    print("converting... ", end = "")
    # ds = convert_units_to(ds, "degC")

    if ds.units == "K":
        ds = (ds - 273.15).assign_attrs(units = "degC")

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # SSTs are stored on a different grid to atmospheric variables - regrid

    # load target grid and cut to the region we're interested in
    psl_fp = glob.glob("/badc/cmip6/data/CMIP6/ScenarioMIP/"+re.sub("\*","psl",m)+"/latest/*")
    if len(psl_fp) == 0: continue
    tmplt = wrap_lon(xr.open_dataset(psl_fp[0])).sel(lon = slice(xn,xx), lat = slice(yn,yx))

    if "i" in ds.dims:   
        # add CF attributes to allow regridding
        ds.i.attrs['axis'] = 'X'
        ds.j.attrs['axis'] = 'Y'

    # build regridder
    print("regridding... ", end = "")
    rg = xe.Regridder(ds, tmplt, "bilinear", ignore_degenerate = True)
    ds_rg = rg(ds).rename("tos").assign_attrs(units = "degC")
    ds_rg = wrap_lon(ds_rg)

    print("saving... ", end = "")
    ds_rg.to_netcdf(new_fnm)
    print("")
clear_output(wait = False)
print("Done.")

# Compile other variables

In [ ]:
for m in mlist:
    
    print(m)
    mdl = m.split("/")[1]+"_"+m.split("/")[3]

    for varnm in ["hus", "ta"]:
        print("  "+varnm, end = ": ")
        
        var_fp = "/badc/cmip6/data/CMIP6/ScenarioMIP/"+re.sub("\*",varnm,m)+"/latest/*"

        fl_hist = glob.glob(re.sub("ScenarioMIP", "CMIP", re.sub("ssp585", "historical", var_fp)))
        fl_ssp = [fnm for fnm in glob.glob(var_fp) if int(fnm[-16:-12]) <= 2100]

        if len(fl_hist) == 0:
            print("No historical data")
            continue
        if len(fl_ssp) == 0: 
            print("No ssp585 data")
            continue

        new_fnm = varnm+"/"+varnm+"_"+rnm+"_"+mdl+fl_hist[0][-17:-9]+fl_ssp[-1].split("/")[-1][-9:]
        if os.path.exists(new_fnm): 
            print("Already processed.")
            continue

        # load & prep the data (cut out small region here)
        print("Loading...", end = " ")
        ds_hist = [wrap_lon(xr.open_dataset(fnm))[varnm].sel(lon = slice(xn, xx), lat = slice(yn,yx)).convert_calendar("standard", align_on = "date") for fnm in fl_hist]
        ds_ssp = [wrap_lon(xr.open_dataset(fnm))[varnm].sel(lon = slice(xn, xx), lat = slice(yn,yx)).convert_calendar("standard", align_on = "date") for fnm in fl_ssp]

        # compile & fix units
        ds = xr.concat(ds_hist + ds_ssp, "time")

        if varnm == "hus" and ds.units == "kg/kg":
            ds = (ds * 100).assign_attrs(units = "%")
            print("Converted", end = "")
        elif varnm == "ta" and ds.units == "K":
            ds = (ds - 273.15).assign_attrs(units = "degC")
            print("Converted", end = "")

        if ds.plev.units == "Pa":
            ds["plev"] = (ds.plev / 100).assign_attrs(units = "hPa")

        ds.to_netcdf(new_fnm)
        print("... Complete.")
    print("")
clear_output(wait = False)
print("Done.")

# Compute potential intensity

In [ ]:
fl = sorted(glob.glob("tos/tos_"+rnm+"*.nc"))
for fnm in fl:

    mdl = "_".join(fnm.split("_")[2:4])
    print(mdl, end = ": ")

    # if mdl in ["FGOALS-g3_r1i1p1f1"]:
    #     print("skipping")
    #     continue

    new_fnm = re.sub("tos","pi",fnm)
    if os.path.exists(new_fnm): 
        print("Already processed")
        continue

    if not all([os.path.exists(re.sub("tos", varnm, fnm)) for varnm in ["hus", "ta", "psl"]]):
        print("Skipped - "+varnm+" not available")
        continue

    print("Loading...", end = " ")
    tos = xr.open_dataset(fnm).tos.sel(lon = slice(xn,xx), lat = slice(yn,yx))
    hus, ta, psl = [xr.open_dataset(re.sub("tos", varnm, fnm))[varnm].sel(lon = slice(xn,xx), lat = slice(yn,yx)) for varnm in ["hus", "ta", "psl"]]

    # merge & compute PI
    ds = xr.merge([tos, psl, ta, hus]).rename(plev = "p", ta = "t", hus = "q", psl = "msl", tos = "sst")

    print("Computing PI...", end = " ")
    # calculate the potential intensity (may take a v long time - up to 3hrs for 200 years)
    vmax, pmin, ifl, t0, otl = xr.apply_ufunc(
        pi,
        ds['sst'], ds['msl'], ds['p'], ds['t'], ds['q'],
        kwargs=dict(CKCD=0.9, ascent_flag=0, diss_flag=1, ptop=50, miss_handle=1),  # use defaults
        input_core_dims=[
            [], [], ['p', ], ['p', ], ['p', ],
        ],
        output_core_dims=[
            [], [], [], [], []
        ],
        vectorize=True
    )

    print("Saving...", end = " ")
    # store the result in an xarray data structure
    ds_out = xr.Dataset({
        'vmax': vmax, 
        'pmin': pmin,
        'ifl': ifl,
        't0': t0,
        'otl': otl,
        })
    
    ds_out.to_netcdf(new_fnm)
    print("Complete.")
# clear_output(wait = False)
print("Done.")